# 05주차 실습: 누수 없는 특성 공학

강의: [PDF](../pdf/week05_feature_engineering.pdf) · **PDF 실제 페이지 5–18** (인쇄 번호와 다를 수 있음)

## 학습 목표
학습 세트 안에서 전처리·변수 선택을 학습하고 검증 성능을 비교한다.

권장 구성: 개념 확인 10분 → 코드 실행 30분 → 변경 실험 30분 → 결과 토의 20분.
설치는 README.md를 따릅니다. 새 커널에서 위에서 아래로 실행하세요.

## 기존 실습 연결
기본 실습 후 [Feature_Engineering_(AI).ipynb](Feature_Engineering_(AI).ipynb)로 이어갑니다. 호환 방법과 수정 사항은 COMPATIBILITY.md를 확인하세요.

In [ ]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / 'labkit').is_dir()), None)
if root is None:
    candidate = here / '강의자료' / '실습코드'
    if candidate.is_dir(): root = candidate
if root is None:
    raise RuntimeError('실습코드 폴더에서 Jupyter를 시작하세요. README.md를 확인하세요.')
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from labkit.common import ensure_data, DATA, output_dir, save_json
ensure_data()

## 기준 모델과 특성 선택

In [ ]:
from labkit.common import ctr_pipeline, split_ctr
from sklearn.metrics import roc_auc_score
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
df = pd.read_csv(DATA/'ctr.csv'); train, valid, test = split_ctr(df)
baseline = ctr_pipeline().fit(train, train.clicked)
selected = ctr_pipeline()
selected.steps.insert(1, ('select', SelectFromModel(LogisticRegression(penalty='l1', solver='liblinear', C=.1, random_state=42))))
selected.fit(train, train.clicked)
results = pd.DataFrame([{'model': name, 'validation_auc': roc_auc_score(valid.clicked, m.predict_proba(valid)[:,1])}
                        for name,m in [('baseline',baseline),('L1 selection',selected)]])
display(results)
out = output_dir('week05'); results.to_csv(out/'feature_comparison.csv', index=False)

## Permutation importance와 PCA

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
importance = permutation_importance(baseline, valid, valid.clicked, scoring='roc_auc', n_repeats=3, random_state=42)
display(pd.Series(importance.importances_mean, index=valid.columns).sort_values(ascending=False))
numeric = ['age','hour','bid_price','ad_quality','popularity']
scaler = StandardScaler().fit(train[numeric])
pca = PCA(n_components=2).fit(scaler.transform(train[numeric]))
print('PCA explained variance:', pca.explained_variance_ratio_)
assert pca.transform(scaler.transform(valid[numeric])).shape == (len(valid),2)
assert 'clicked' not in baseline['pre'].get_feature_names_out().tolist()
print('test는 아직 모델 선택에 사용하지 않았습니다.')

## 확장 과제 및 제출
결측값을 주입하고 imputer 전략 및 L1 강도를 바꿔 검증 AUC·선택 변수 수를 비교하세요. 기존 회귀 실습과 분류 실습의 중요도·평가 지표를 구분하세요.

제출: 실행한 노트북, 결과 표/그림, 변경한 조건과 해석. outputs/weekXX에 저장된 자료를 첨부하세요.

평가 기준: 개념 연결 25%, 실행·재현성 30%, 비교 실험 25%, 해석·한계 20%.